# Otimização com PySpark

## Importações e criação da sessão Spark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Otimizacao").getOrCreate()


## Leitura dos arquivos parquet

In [ ]:
# Leitura do dataframe de vídeos preparados
df_video = spark.read.parquet("videos-preparados.snappy.parquet")

# Leitura do dataframe de comentários tratados
df_comments = spark.read.parquet("videos-comments-tratados.snappy.parquet")

# Visualização inicial
df_video.show(5)
df_comments.show(5)


## Criação de tabelas temporárias

In [ ]:
# Criando views temporárias para uso no Spark SQL
df_video.createOrReplaceTempView("tb_video")
df_comments.createOrReplaceTempView("tb_comments")


## Join utilizando Spark SQL

In [ ]:
# Realizando join entre vídeos e comentários
join_video_comments = spark.sql("""
SELECT *
FROM tb_video v
INNER JOIN tb_comments c
ON v.`Video ID` = c.`Video ID`
""")

join_video_comments.show(5)


## Utilizando repartition e coalesce

In [ ]:
# Repartition redistribui os dados aumentando paralelismo
df_video_rep = df_video.repartition(4, "Video ID")

# Coalesce reduz quantidade de partições
df_comments_coalesce = df_comments.coalesce(2)

# Criando novas views temporárias
df_video_rep.createOrReplaceTempView("tb_video_rep")
df_comments_coalesce.createOrReplaceTempView("tb_comments_coalesce")

# Join utilizando Spark SQL novamente
join_video_comments_2 = spark.sql("""
SELECT *
FROM tb_video_rep v
INNER JOIN tb_comments_coalesce c
ON v.`Video ID` = c.`Video ID`
""")

join_video_comments_2.show(5)


## Explain dos planos de execução

In [ ]:
# Plano de execução do primeiro join
join_video_comments.explain()

# Plano de execução do segundo join
join_video_comments_2.explain()


## Join otimizado com seleção e filtro

In [ ]:
# Selecionando apenas colunas necessárias
df_video_otimizado = df_video.select(
    "Video ID",
    "Title",
    "Views",
    "Likes",
    "Year"
)

df_comments_otimizado = df_comments.select(
    "Video ID",
    "Comment"
)

# Filtrando apenas vídeos com Views maiores que 1000
df_video_otimizado = df_video_otimizado.filter(df_video_otimizado["Views"] > 1000)

# Repartitionando pelo campo de join
df_video_otimizado = df_video_otimizado.repartition(4, "Video ID")

# Realizando join otimizado
join_otimizado = df_video_otimizado.join(
    df_comments_otimizado,
    on="Video ID",
    how="inner"
)

# Visualizando resultado
join_otimizado.show(5)

# Plano de execução otimizado
join_otimizado.explain()


## Salvando parquet otimizado

In [ ]:
# Salvando resultado final em parquet
join_otimizado.write.mode("overwrite").parquet("join-videos-comments-otimizado")
